Load the file and check the data making sure there are not a lot of null values

In [ ]:
import pandas as pd
df = pd.read_json("../data/raw/dramas.jsonl", lines=True)
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = pd.read_json("../data/raw/tvshows.jsonl", lines=True)
df["country"].value_counts()

In [ ]:
paths = {
    "drama": "../data/raw/dramas.jsonl",
    "movie": "../data/raw/movies.jsonl",
    "tvshow": "../data/raw/tvshows.jsonl",
    "special": "../data/raw/specials.jsonl",
}

dfs = []
for content_type, path in paths.items():
    d = pd.read_json(path, lines=True)
    d["content_type"] = content_type
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)

Clean the data to be Chinese shows only and grouping all content types into a single dataFrame

In [ ]:
df = df[df["country"] == "China"].copy()

Checking whether there are duplicates title and we should check for duplicates by ID

In [ ]:
df["title_str"] = df["titles"].apply(lambda t: t.get("english") or t.get("native"))
dupe_titles = df[df.duplicated(subset="title_str", keep=False)]
dupe_titles[["title_str", "content_type", "id"]].sort_values("title_str")

In [ ]:
import pandas as pd
df = pd.read_pickle("../data/processed/clean_dramas.pkl")
print(df["soup"].iloc[0])
print(df["content_type"].value_counts())

Make sure that cosine similarity between two similar shows should be decently high

Making sure that two seperate dramas that are unrelated have low score

In [ ]:
import sys
sys.path.append("../src")
from recommendation.similarity import load_data, load_model, search

df, embeddings = load_data()
model = load_model()

search("modern college romance", df, embeddings, model, k=10)

In [ ]:
import importlib
import data.mdl_client
importlib.reload(data.mdl_client)
from data.mdl_client import search_title, get_slug, get_recommendations, build_relevant_list


slug = get_slug("Nirvana in Fire")
print(slug)

Set of draft_qeuries

In [ ]:
draft_queries = [
    {"query": "historical romance with political intrigue", "seed_title": "Nirvana in Fire"},
    {"query": "wuxia revenge story", "seed_title": "The Legend of the Condor Heroes"},
    {"query": "modern college romance", "seed_title": "A Love So Beautiful"},
    {"query": "workplace romance with lots of comedy", "seed_title": "Love O2O"},
    {"query": "Chinese crime mystery", "seed_title": "The Long Night"},
    {"query": "family drama about complicated relationships between siblings", "seed_title": "Go Ahead"},
    {"query": "fantasy romance with gods and immortals", "seed_title": "Eternal Love"},
    {"query": "coming-of-age story about friendship and first love", "seed_title": "When We Were Young"},
    {"query": "revenge thriller", "seed_title": "The Double"},
    {"query": "quiet slice-of-life drama about ordinary people", "seed_title": "Minning Town"},
    {"query": "historical court drama with power struggles", "seed_title": "The Longest Day in Chang'an"},
    {"query": "martial arts adventure", "seed_title": "Mysterious Lotus Casebook"},
    {"query": "sweet modern romance", "seed_title": "Hidden Love"},
    {"query": "office romance between coworkers who slowly fall in love", "seed_title": "The Rational Life"},
    {"query": "detective mystery with a dark atmosphere", "seed_title": "Under the Skin"},
    {"query": "modern family drama about relationships, careers, and friendship", "seed_title": "Ode to Joy"},
    {"query": "xianxia fantasy with tragic romance", "seed_title": "Love Between Fairy and Devil"},
    {"query": "high school friendship", "seed_title": "With You"},
    {"query": "historical revenge and political conspiracy", "seed_title": "The Rise of Phoenixes"},
    {"query": "small-town slice of life with romance", "seed_title": "Meet Yourself"},
    {"query": "wuxia mystery with a group of unlikely heroes", "seed_title": "Side Story of Fox Volant"},
    {"query": "romantic comedy about two people reconnecting after school", "seed_title": "You Are My Glory"},
    {"query": "intense crime thriller about a serial killer", "seed_title": "Burning Ice"},
    {"query": "sweet high school romance with friendship and youthful coming-of-age", "seed_title": "When I Fly Towards You"},
    {"query": "historical drama about rival kingdoms, military strategy, and complicated loyalties", "seed_title": "The Long Ballad"},
    {"query": "fantasy adventure with demons, magic, and a slow-burn romance", "seed_title": "The Untamed"},
    {"query": "emotional family story centered on three unrelated children growing up together", "seed_title": "The Bond"},
    {"query": "dark mystery where the investigation gradually uncovers a much larger conspiracy", "seed_title": "The Bad Kids"},
    {"query": "lighthearted romance with an awkward but lovable male lead", "seed_title": "Put Your Head on My Shoulder"},
    {"query": "romantic comedy about two people from very different backgrounds", "seed_title": "My Little Happiness"},
]

In [ ]:
missing = []
for item in draft_queries:
    match = df[df["soup"].str.contains(f"Title: {item['seed_title']}\n", regex=False)]
    if match.empty:
        missing.append(item["seed_title"])

print(f"{len(missing)} missing out of {len(draft_queries)}")
print(missing)

In [ ]:
import time
import json

all_queries = []
for item in draft_queries:
    try:
        relevant = build_relevant_list(item["seed_title"], df)
    except Exception as e:
        print(f"Failed for {item['seed_title']}: {e}")
        relevant = []
    all_queries.append({
        "query": item["query"],
        "seed_title": item["seed_title"],
        "relevant": relevant
    })
    time.sleep(1)

with open("../data/processed/queries.json", "w") as f:
    json.dump(all_queries, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_queries)} queries")

In [ ]:
for item in all_queries:
    print(f"{item['query']}: {len(item['relevant'])} relevant titles")

In [ ]:
zero_seeds = ["The Long Night", "Eternal Love", "When We Were Young", "The Double",
              "Minning Town", "Hidden Love", "The Rise of Phoenixes", "Side Story of Fox Volant",
              "The Bad Kids"]  # your actual 9 zero seeds

for seed in zero_seeds:
    slug = get_slug(seed)
    recs = get_recommendations(slug)
    print(f"{seed} -> slug={slug}, total_recs={recs.get('total')}")

In [ ]:
result = search_title("The Bad Kids")
for m in result["results"]:
    print(m["title"], "|", m["slug"], "|", m.get("year"))

In [ ]:
results = search("historical romance with political intrigue", df, embeddings, model, k=10)
print(results[["soup", "similarity"]].apply(lambda r: (r["soup"].split("\n")[0], r["similarity"]), axis=1).tolist())

In [ ]:
from recommendation.llm import expand_query

for q in ["historical romance with political intrigue", "detective mystery with a dark atmosphere", "wuxia mystery with a group of unlikely heroes"]:
    print(f"ORIGINAL: {q}")
    print(f"EXPANDED: {expand_query(q)}")
    print()

In [ ]:
import importlib
import recommendation.llm
importlib.reload(recommendation.llm)
from recommendation.llm import extract_preferences

for q in ["historical romance with political intrigue", "detective mystery with a dark atmosphere", "wuxia mystery with a group of unlikely heroes"]:
    print(q, "->", extract_preferences(q))

In [1]:
import pandas as pd
df = pd.read_pickle("../data/processed/clean_dramas.pkl")

In [2]:
df.iloc[0][["id", "genres", "tags", "cover", "date"]]

id                                                     9025
genres    [{'name': 'Military', 'id': '3'}, {'name': 'Hi...
tags      [{'name': 'Power Struggle', 'id': '1502'}, {'n...
cover              https://i.mydramalist.com/kV54dc.jpg?v=1
date                                    2015-09-19 00:00:00
Name: 0, dtype: object

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_pickle("../data/processed/clean_dramas.pkl")
embeddings = np.load("../data/processed/embeddings.npy")

nif_idx = df[df["soup"].str.contains("Title: Nirvana in Fire\n", regex=False)].index[0]

sims = cosine_similarity(embeddings[[nif_idx]], embeddings)[0]
df["sim_to_nif"] = sims

print(df.sort_values("sim_to_nif", ascending=False)[["soup", "sim_to_nif"]].head(15))

                                                   soup  sim_to_nif
0     Title: Nirvana in Fire\nType: drama\nGenres: M...    1.000000
39    Title: Nirvana in Fire Season 2: The Wind Blow...    0.718376
199   Title: Love of Nirvana\nType: drama\nGenres: R...    0.712847
347   Title: Royal Nirvana Special\nType: drama\nGen...    0.686448
315   Title: Royal Nirvana\nType: drama\nGenres: His...    0.659382
911   Title: Returned Master\nType: drama\nGenres: W...    0.654776
807   Title: Colourful Bone\nType: drama\nGenres: Th...    0.641593
1374  Title: God of Lost Fantasy\nType: drama\nGenre...    0.635396
273   Title: Youthful Glory\nType: drama\nGenres: Hi...    0.625187
897   Title: The Promise of Chang’an\nType: drama\nG...    0.624008
463   Title: Burning Flames\nType: drama\nGenres: Wu...    0.623227
68    Title: The Longest Day in Chang'an\nType: dram...    0.619606
1213  Title: Untouchable Lovers\nType: drama\nGenres...    0.613405
824   Title: Renascence\nType: drama\nGenres: Hi

In [15]:
import os
import matplotlib.pyplot as plt
import numpy as np

# Get the directory containing this script
BASE_DIR = os.path.dirname(os.path.abspath("assets/eval_chart.png"))
ASSETS_DIR = os.path.join(BASE_DIR, "assets")

os.makedirs(ASSETS_DIR, exist_ok=True)

# Data
models = [
    'all-mpnet-base-v2\n(Local / Heavy)',
    'all-MiniLM-L6-v2\n(Deployed / Light)'
]

baseline_p5 = [0.096, 0.040]
hybrid_p5 = [0.120, 0.072]

x = np.arange(len(models))
width = 0.35

# Figure
fig, ax = plt.subplots(figsize=(9, 5), dpi=300)

color_baseline = '#4682B4'
color_hybrid = '#E67E22'

rects1 = ax.bar(
    x - width / 2,
    baseline_p5,
    width,
    label='Baseline (Cosine Similarity)',
    color=color_baseline,
    edgecolor='black',
    linewidth=1
)

rects2 = ax.bar(
    x + width / 2,
    hybrid_p5,
    width,
    label='Hybrid (Groq LLM Re-rank)',
    color=color_hybrid,
    edgecolor='black',
    linewidth=1
)

# Labels
ax.set_ylabel(
    'Precision@5 Score',
    fontsize=11,
    fontweight='bold'
)

ax.set_title(
    'YouYuan Search Benchmark: Baseline Vector Search vs. Hybrid Re-ranking',
    fontsize=12,
    fontweight='bold',
    pad=15
)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)

ax.legend(
    frameon=True,
    facecolor='white',
    edgecolor='none'
)

ax.set_ylim(0, 0.15)

ax.grid(
    axis='y',
    linestyle='--',
    alpha=0.5
)

# Add values above bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()

        ax.annotate(
            f'{height:.3f}',
            xy=(
                rect.get_x() + rect.get_width() / 2,
                height
            ),
            xytext=(0, 3),
            textcoords="offset points",
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )

autolabel(rects1)
autolabel(rects2)

# Save
from pathlib import Path

# Project root = one level above notebooks/
PROJECT_ROOT = Path.cwd().parent

# Root-level assets folder
ASSETS_DIR = PROJECT_ROOT / "assets"
ASSETS_DIR.mkdir(exist_ok=True)

output_path = ASSETS_DIR / "eval_chart.png"

plt.tight_layout()

plt.savefig(
    output_path,
    format="png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print("Chart generated successfully:")
print(output_path)
print(f"File exists: {output_path.exists()}")

Chart generated successfully:
/Users/dannyzheng/YouYuan/assets/eval_chart.png
File exists: True
